# RSNA 2022 PED: Drive zip -> GCS

这个笔记本用于：
- 挂载 Google Drive
- 认证 GCS
- 将 Drive 中的 zip 解压后上传到 GCS 指定目录

你只需要修改第一个代码单元的超参即可复用。

In [4]:
# Cell 1: 超参配置（后续修改只改这里）
PROJECT_ID = "brats-preprocess"
GCS_BUCKET = "rsna_2022_ped_preprocessed_npy"

# 显式超参：zip 文件名 与 GCS 目标文件夹名
ZIP_FILE_NAME = "npy2.zip"
TARGET_GCS_FOLDER = "npy2"

# 每批上传文件数（你要求改成 100）
BATCH_UPLOAD_SIZE = 1000

# 你的 Drive 内 zip 所在路径
DRIVE_ZIP_PATH = f"/content/drive/MyDrive/dataset/pretrain/RSNA-2022-PED/{ZIP_FILE_NAME}"

# 上传目标前缀
GCS_TARGET_PREFIX = f"gs://{GCS_BUCKET}/{TARGET_GCS_FOLDER}"

print("Configuration:")
print(f"- PROJECT_ID: {PROJECT_ID}")
print(f"- GCS_BUCKET: {GCS_BUCKET}")
print(f"- ZIP_FILE_NAME: {ZIP_FILE_NAME}")
print(f"- TARGET_GCS_FOLDER: {TARGET_GCS_FOLDER}")
print(f"- BATCH_UPLOAD_SIZE: {BATCH_UPLOAD_SIZE}")
print(f"- DRIVE_ZIP_PATH: {DRIVE_ZIP_PATH}")
print(f"- GCS_TARGET_PREFIX: {GCS_TARGET_PREFIX}")

Configuration:
- PROJECT_ID: brats-preprocess
- GCS_BUCKET: rsna_2022_ped_preprocessed_npy
- ZIP_FILE_NAME: npy2.zip
- TARGET_GCS_FOLDER: npy2
- BATCH_UPLOAD_SIZE: 100
- DRIVE_ZIP_PATH: /content/drive/MyDrive/dataset/pretrain/RSNA-2022-PED/npy2.zip
- GCS_TARGET_PREFIX: gs://rsna_2022_ped_preprocessed_npy/npy2


In [5]:
# Cell 2: 挂载 Drive + 认证 GCS
from google.colab import drive, auth

drive.mount('/content/drive')
auth.authenticate_user()

import os
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

!gcloud config set project {PROJECT_ID}
!gcloud config get-value project
!gsutil ls -L -b gs://{GCS_BUCKET}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
brats-preprocess
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://rsna_2022_ped_preprocessed_npy/ :
	Storage class:			STANDARD
	Location type:			multi-region
	Location constraint:		US
	Versioning enabled:		None
	Logging configuration:		None
	Website configuration:		None
	CORS configuration: 		None
	Lifecycle configuration:	None
	Requester Pays enabled:		None
	Labels:				None
	Default KMS key:		None
	Time created:			Mon, 20 Apr 2026 18:45:38 GMT
	Time updated:			Mon, 20 Apr 2026 18:45:38 GMT
	Metageneration:			1
	Bucket Policy Only enabled:	True
	P

In [6]:
# Cell 3: 每批解压并并行上传到 GCS（默认每批 100 个文件）
import os
import zipfile
import subprocess
import tempfile
import shutil

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(f"Zip not found: {DRIVE_ZIP_PATH}")

batch_size = max(1, int(BATCH_UPLOAD_SIZE))

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    file_infos = [info for info in zf.infolist() if not info.is_dir()]

    print(f"Zip file: {DRIVE_ZIP_PATH}")
    print(f"Total files to upload: {len(file_infos)}")
    print(f"Batch size: {batch_size}")

    # 预览前几个条目
    for info in file_infos[:10]:
        print(" -", info.filename)

    uploaded = 0
    total = len(file_infos)

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = file_infos[start:end]

        with tempfile.TemporaryDirectory(prefix="zip_batch_") as batch_dir:
            for info in batch:
                rel_path = info.filename.lstrip('/').replace('\\', '/')
                local_path = os.path.join(batch_dir, rel_path)
                os.makedirs(os.path.dirname(local_path), exist_ok=True)

                with zf.open(info, 'r') as src, open(local_path, 'wb') as dst:
                    shutil.copyfileobj(src, dst, length=1024 * 1024)

            cmd = f'gsutil -m cp -r "{batch_dir}/." "{GCS_TARGET_PREFIX}/"'
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(
                    "Batch upload failed\n"
                    f"batch range: {start + 1}-{end}\n"
                    f"stdout: {result.stdout}\n"
                    f"stderr: {result.stderr}"
                )

        uploaded = end
        print(f"Uploaded {uploaded}/{total} (batch {start + 1}-{end})")

print(f"Batch upload finished. Uploaded files: {uploaded}")

Zip file: /content/drive/MyDrive/dataset/pretrain/RSNA-2022-PED/npy2.zip
Total files to upload: 23308
Batch size: 100
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_35.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_36.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_37.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_38.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_39.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_4.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_40.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_41.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_42.npy
 - npy2/RSNA-PED_train_part1_1a99611b221f_4fa5e4d4f9f3_0_43.npy
Uploaded 100/23308 (batch 1-100)


KeyboardInterrupt: 

In [ ]:
# Cell 4: 上传结果检查
print("Sample objects under target prefix:")
!gsutil ls {GCS_TARGET_PREFIX}/ | head -20

print("\nSample .npy objects:")
!gsutil ls -r {GCS_TARGET_PREFIX}/**/*.npy | head -20

In [3]:
# Cell 5: 统计 npy 文件数量（可选）
!gsutil ls -r {GCS_TARGET_PREFIX}/**/*.npy | wc -l

23308
